In [ ]:
%%capture
!pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.0.post2 --no-build-isolation
!pip install evo2

In [ ]:
import os

WORK_DIR = "/content"
CACHE_DIR = "/content/hf"

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}"

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import torch
from evo2 import Evo2

evo2_model = Evo2('evo2_7b')

sequence = 'ACGT'
input_ids = torch.tensor(
    evo2_model.tokenizer.tokenize(sequence),
    dtype=torch.int,
).unsqueeze(0).to('cuda:0')

outputs, _ = evo2_model(input_ids)
logits = outputs[0]

print('Logits: ', logits)
print('Shape (batch, length, vocab): ', logits.shape)

In [ ]:
import torch
from evo2 import Evo2

evo2_model = Evo2('evo2_7b')

sequence = 'ACGT'
input_ids = torch.tensor(
    evo2_model.tokenizer.tokenize(sequence),
    dtype=torch.int,
).unsqueeze(0).to('cuda:0')

layer_name = 'blocks.28.mlp.l3'

outputs, embeddings = evo2_model(input_ids, return_embeddings=True, layer_names=[layer_name])

print('Embeddings shape: ', embeddings[layer_name].shape)

In [ ]:
!gsutil ls gs://vfdb/vfdb_results_filter1/** | wc -l

In [ ]:
!mkdir -p /home/jupyter/data
!gsutil -m cp -r gs://vfdb/vfdb_results_filter1 /home/jupyter/data/

In [ ]:
import os

INPUT_DIR = "/home/jupyter/data/vfdb_results_filter1"
print("Number of files:", len(os.listdir(INPUT_DIR)))
print(os.listdir(INPUT_DIR)[:5])

In [ ]:
OUTPUT_DIR = "/home/jupyter/data/evo2_embeddings_filter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def get_embedding(sequence, model, layer_name):
    tokens = model.tokenizer.tokenize(sequence)

    input_ids = torch.tensor(tokens, dtype=torch.int).unsqueeze(0).to(device)

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[layer_name]
        )

    emb = embeddings[layer_name].squeeze(0).cpu()

    #  mean pooling
    emb = emb.mean(dim=0)

    del input_ids, embeddings
    torch.cuda.empty_cache()

    return emb

In [ ]:
from Bio import SeqIO

def process_fasta(fasta_path, output_path, model, layer_name):
    all_embeddings = []

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq)

        # safety limit
        seq = seq[:2000]

        try:
            emb = get_embedding(seq, model, layer_name)
            all_embeddings.append(emb)

        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM → skipping {record.id}")
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    torch.save(all_embeddings, output_path)

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

In [ ]:
from tqdm import tqdm

layer_name = 'blocks.28.mlp.l3'

fasta_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".fasta")]

for fasta_file in tqdm(fasta_files):
    input_path = os.path.join(INPUT_DIR, fasta_file)
    output_path = os.path.join(OUTPUT_DIR, fasta_file.replace(".fasta", ".pt"))

    if os.path.exists(output_path):
        continue

    process_fasta(input_path, output_path, evo2_model, layer_name)

In [ ]:
!gsutil -m cp -r /home/jupyter/data/evo2_embeddings_filter gs://vfdb/

In [ ]:
import torch
import os

emb_path = "/home/jupyter/data/evo2_embeddings_filter"

files = [f for f in os.listdir(emb_path) if f.endswith(".pt")]

sample_file = os.path.join(emb_path, files[0])

embeddings = torch.load(sample_file)

print("Number of sequences:", len(embeddings))
print("Embedding shape (first seq):", embeddings[0].shape)

In [ ]:
import os

INPUT_ = "/home/jupyter/data/evo2_embeddings_filter"
print("Number of files:", len(os.listdir(INPUT_)))
print(os.listdir(INPUT_)[:5])